# Stock Price Prediction with LSTM — Quick Start

This notebook will guide you through:
1. Loading your own stock data
2. Training an LSTM model
3. Making predictions
4. Visualizing the results

**Requirements:** Your CSV file must have columns: `Date`, `Open`, `High`, `Low`, `Close`

In [ ]:
# Import the library
from stock_market_lstm.dataset import load_from_csv
from stock_market_lstm.features import StockPreprocessor
from stock_market_lstm.modeling import LSTMPredictor
from stock_market_lstm import plots

import matplotlib.pyplot as plt
import numpy as np

# Set seeds for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

print("✅ All imports successful!")

### 1. Load Your Data

Change the path below to point to your CSV file.

In [ ]:
# Load your data
DATA_PATH = "examples/sample_data.csv"  # Change this to your file!

df = load_from_csv(DATA_PATH)
print(f"Loaded {len(df)} trading days")
print(f"Date range: {df['Date'].iloc[0]} to {df['Date'].iloc[-1]}")
df.head()

In [ ]:
# Visualize raw data
plots.plot_raw_prices(df)

### 2. Preprocess Data

The preprocessor handles:
- Splitting into train/test (80/20 by default)
- MinMax scaling
- EMA smoothing (reduces noise in training data)

In [ ]:
# Preprocess the data
preprocessor = StockPreprocessor(test_size=0.2, ema_gamma=0.1)
data = preprocessor.process(df)

print(f"Training data:   {len(data['train'])} points")
print(f"Test data:       {len(data['test'])} points")
print(f"Combined:        {len(data['full'])} points")

### 3. Build and Train the LSTM Model

In [ ]:
# Create and train the model
model = LSTMPredictor(seq_len=50)
model.build()

# Train (more epochs = potentially better, but slower)
history = model.train(data['train'], epochs=30, verbose=1)

# Plot training progress
plots.plot_training_history(history.history)

### 4. Make Predictions

In [ ]:
# Generate predictions on test data
full_series = data['full']
context = 50
steps = 50
start_index = len(full_series) - steps - 1

predictions = model.recursive_forecast(
    series=full_series,
    start_index=start_index,
    context=context,
    steps=steps
)

print(f"Generated {len(predictions)} predictions")
print(f"First 5 predictions: {predictions[:5]}")

### 5. Visualize Results

In [ ]:
# Compare predictions to actual values
plots.plot_predictions_vs_true(
    true_values=full_series,
    predictions=predictions,
    offset=start_index,
    title="LSTM Predictions vs Actual Prices"
)

### 6. Save Your Model

Save the trained model to use later without retraining.

In [ ]:
# Save the trained model
model.save("models/my_trained_model.h5")
print("Model saved! You can load it later with:")
print('  model = LSTMPredictor().load("models/my_trained_model.h5")')

### Next Steps

- Try different `seq_len` values (30, 50, 100) to see how context length affects predictions
- Adjust `epochs` — more epochs may improve results but risk overfitting
- Experiment with `ema_gamma` in the preprocessor for different smoothing levels
- Use your own CSV file with real stock data!